# SuperLLM 증류 — Colab (무료 GPU, 세션 분할 실행)

**Gemma 4 E4B(교사, 4bit) → LFM2.5-350M(학생, QLoRA) ULD 증류.**

무료 Colab은 세션이 끊기므로(유휴 ~90분, 일일 GPU 쿼터) **체크포인트를 Google
Drive에 저장**하고 나눠서 실행한다. 학습 셀(Step 5)을 **매 세션 다시 실행하면**
`trainer_state.json`을 읽어 **자동으로 이어서** 학습한다.

> 런타임 유형을 **GPU(T4)**로: 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU.

In [ ]:
!nvidia-smi -L  # GPU 확인 (없으면 런타임 유형을 T4로 변경)

## Step 1 — Google Drive 마운트 (체크포인트 영속화)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — 리포 클론 + 의존성 설치

> 비공개 리포면 클론에 인증이 필요하다. 토큰을 쓰거나 `distill/` 폴더를 직접
> 업로드해도 된다.

In [ ]:
%cd /content
!git clone -b claude/sub-500mb-llm-sa3xbc https://github.com/operator0225/SuperLLM.git \
  || (cd SuperLLM && git pull)
%cd /content/SuperLLM
!pip -q install -r distill/requirements.txt

## Step 3 — 체크포인트를 Drive로 지정

`output_dir`/`target_data`를 Drive 경로로 바꿔 세션이 끊겨도 유지되게 한다.

In [ ]:
import yaml
CKPT_DIR = '/content/drive/MyDrive/superllm_ckpt/lfm2.5-350m-uld'  # Drive → 세션 넘어 유지
DATA     = '/content/drive/MyDrive/superllm_ckpt/target_data.jsonl'
p = 'distill/config.yaml'
c = yaml.safe_load(open(p))
dl = c['distill_logit']
dl['output_dir']  = CKPT_DIR
dl['target_data'] = DATA
dl['teacher_model'] = 'google/gemma-4-E4B-it'  # 정확한 repo id는 HF에서 확인
dl['teacher_precision'] = 'nf4'                # T4 16GB → 교사 NF4
dl['student_load_in_4bit'] = True              # 학생 QLoRA
dl['save_steps'] = 100                         # 자주 저장(끊김 대비)
yaml.safe_dump(c, open(p, 'w'), allow_unicode=True)
print('output_dir →', CKPT_DIR)

## Step 4 — (옵션) 증류 대상 텍스트 준비

ULD는 교사·학생을 같은 텍스트에 통과시킨다. 대상 텍스트가 없으면 작은 공개
instruct 데이터로 만든다. (이미 있으면 이 셀은 건너뜀.)

In [ ]:
import os, json
if not os.path.exists(DATA):
    from datasets import load_dataset
    ds = load_dataset('databricks/databricks-dolly-15k', split='train[:2000]')
    os.makedirs(os.path.dirname(DATA), exist_ok=True)
    n = 0
    with open(DATA, 'w') as f:
        for r in ds:
            ins, resp = r.get('instruction','').strip(), r.get('response','').strip()
            if ins and resp:
                f.write(json.dumps({'prompt': ins, 'response': resp}, ensure_ascii=False) + '\n')
                n += 1
    print('wrote', n, '→', DATA)
else:
    print('already exists →', DATA)

## Step 5 — ULD 증류 학습  ⭐ 매 세션 이 셀을 다시 실행

세션이 끊기면 Step 1~3을 다시 실행한 뒤 **이 셀만 재실행**하면 마지막
체크포인트부터 이어진다. (`[resume] ... 부터 재개` 로그 확인.)

In [ ]:
!python distill/train_distill_logit.py --config distill/config.yaml

## Step 6 — 배포용 GGUF q4_k_m 변환

학습이 끝나면 LoRA 어댑터를 base에 병합 → GGUF → q4_k_m. 상세 명령은
`distill/export_gguf.md` 참고. 결과 ~200MB 파일 하나만 S25+로 옮기면 된다.

In [ ]:
# 요약: (1) PeftModel.merge_and_unload 로 fp16 병합  (2) convert_hf_to_gguf.py
#       (3) llama-quantize ... Q4_K_M   → distill/export_gguf.md 참조
print('학습 완료 후 distill/export_gguf.md 의 병합+변환 단계를 따르세요.')